In [55]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import pickle
import os 
import warnings
warnings.filterwarnings('ignore')

In [56]:
dataset = pd.read_csv("../data/amazon_products.csv", nrows=10000)
print(f"===Dataset Loaded===")
print(f"Columns: {dataset.columns.to_list()}")

===Dataset Loaded===
Columns: ['asin', 'title', 'imgUrl', 'productURL', 'stars', 'reviews', 'price', 'listPrice', 'category_id', 'isBestSeller', 'boughtInLastMonth']


In [57]:
# Loading Content Model
from sklearn.metrics.pairwise import cosine_similarity
try:
    with open('../artifacts/tfidf_vectorizer.pkl', 'rb') as f:
        vectorizer = pickle.load(f)
    print("===Loaded TF-IDF vectorizer===")

except:
    print("Vectorizer Not found Creating new one...")
    vectorizer = TfidfVectorizer(stop_words='english', max_features=2000)

# Create product features
def create_text(row):
    features = [str(row['title']), f"category_{row['category_id']}"]
    if row['price'] > 0:
        if row['price'] < 50:
            features.append("Budget")
        elif row['price'] < 200:
            features.append("Mid-range")
        else:
            features.append("Premium")
    
    if row['isBestSeller']:
        features.append("bestseller")
    return " ".join(features)

dataset['product_text'] = dataset.apply(create_text, axis = 1)

# Create TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(dataset['product_text'])

print(f" TF-IDF matrix shape: {tfidf_matrix.shape}")
print("Contel Model is ready - using Cosine Similarity")


===Loaded TF-IDF vectorizer===
 TF-IDF matrix shape: (10000, 5000)
Contel Model is ready - using Cosine Similarity


In [58]:
# Calculating popularity scores 
max_bought = dataset['boughtInLastMonth'].max()
dataset['popularity_score'] = dataset['boughtInLastMonth'] / max_bought if max_bought > 0 else 0

print(" Popularity scores calculated")
print(f"   Range: {dataset['popularity_score'].min():.2f} to {dataset['popularity_score'].max():.2f}")

 Popularity scores calculated
   Range: 0.00 to 1.00


In [59]:
# Create content function using cosine similarity
def get_content_similarity(product_idx):
    """Get content-based similarity scores using cosine_similarity"""
    content_scores = cosine_similarity(tfidf_matrix[product_idx], tfidf_matrix)[0]
    # Get top 51 indices (including itself)
    indices = content_scores.argsort()[::-1][:51]
    scores = content_scores[indices]
    return indices, scores
def recommend_content(dataset, product_name, n=5):
    """Pure content recommendation"""
    matching = dataset[dataset['title'].str.contains(product_name, case=False)]
    if len(matching) == 0:
        return f"Product '{product_name}' not found"
    
    product_idx = matching.index[0]
    indices, scores = get_content_similarity(product_idx)
    
    recommendations = []
    for i in range(1, len(indices)):
        idx = indices[i]
        recommendations.append({
            'title': dataset.iloc[idx]['title'],
            'price': dataset.iloc[idx]['price'],
            'stars': dataset.iloc[idx]['stars'],
            'similarity': round(scores[i], 3)
        })
    
    return pd.DataFrame(recommendations[:n])

print(" Content Model ready - using cosine_similarity)")
print(f"Test content recommendation: {recommend_content(dataset, 'Luggage', 3).iloc[0]['title'][:40] if len(recommend_content(data, 'Luggage', 3)) > 0 else 'N/A'}...")

 Content Model ready - using cosine_similarity)


NameError: name 'data' is not defined

In [ ]:
# Hybrid Recommender- combines both - Popularity recommender and content based recommender
def hybrid_recommend(product_name, n=5, content_weight=0.7, popularity_weight=0.3):
    """Combine content + popularity scores"""
    
    matching = dataset[dataset['title'].str.contains(product_name, case=False)]
    if len(matching) == 0:
        return f"Product '{product_name}' not found"
    
    product_idx = matching.index[0]
    
    # Get content scores
    content_scores = cosine_similarity(tfidf_matrix[product_idx], tfidf_matrix)[0]
    
    # Calculate hybrid scores for all products
    recommendations = []
    for idx in range(len(dataset)):
        if idx != product_idx:
            content_score = content_scores[idx]
            popularity_score = dataset.iloc[idx]['popularity_score']
            hybrid_score = (content_weight * content_score) + (popularity_weight * popularity_score)
            
            recommendations.append({
                'title': dataset.iloc[idx]['title'],
                'price': dataset.iloc[idx]['price'],
                'stars': dataset.iloc[idx]['stars'],
                'bought': dataset.iloc[idx]['boughtInLastMonth'],
                'content_score': round(content_score, 3),
                'popularity_score': round(popularity_score, 3),
                'hybrid_score': round(hybrid_score, 3)
            })
    
    recommendations.sort(key=lambda x: x['hybrid_score'], reverse=True)
    return pd.DataFrame(recommendations[:n])

print(" Hybrid Recommender ready!")

 Hybrid Recommender ready!


In [ ]:
# Testing  Hybrid Recommender model
print("="*60)
print(" TEST: Recommendations for 'Luggage'")
print("="*60)

results = hybrid_recommend("Luggage", 5)

if isinstance(results, pd.DataFrame):
    for i, row in results.iterrows():
        print(f"\n{i+1}. {row['title'][:60]}...")
        print(f"  ${row['price']} |  {row['stars']}")
        print(f"  Hybrid Score: {row['hybrid_score']}")
        print(f"  (Content: {row['content_score']} + Popularity: {row['popularity_score']})")
else:
    print(results)

 TEST: Recommendations for 'Socks'

1. Ascella X Softside Expandable Luggage with Spinners, Black, ...
  $198.41 |  4.5
  Hybrid Score: 0.412
  (Content: 0.584 + Popularity: 0.01)

2. 28 Inch Luggage with Spinner Wheels Softside Expandable Larg...
  $149.99 |  0.0
  Hybrid Score: 0.331
  (Content: 0.473 + Popularity: 0.0)

3. Cambridge Lightweight Luggage Softside Expandable Spinner Wh...
  $164.99 |  4.3
  Hybrid Score: 0.319
  (Content: 0.455 + Popularity: 0.0)

4. Anzio Softside Expandable Spinner Luggage, Teal, Checked-Lar...
  $98.62 |  4.1
  Hybrid Score: 0.31
  (Content: 0.43 + Popularity: 0.03)

5. Bromley Softside Expandable Spinner Luggage, Black, Checked-...
  $172.19 |  4.5
  Hybrid Score: 0.309
  (Content: 0.441 + Popularity: 0.0)
